# Auto-Discover -> Inject -> Verify

1. Pick a downstream agent
2. **Preflight** - check & install agent dependencies
3. Scan source code for tool registration patterns
4. Generate wrappers and inject discovered tools
5. Verify every injected tool can actually be called

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "requests", "beautifulsoup4", "pyyaml", "openai",
                "graph-tool-call", "pandas", "langchain-openai"],
               check=True)
from IPython.display import display, Markdown
display(Markdown("**deps installed**"))

In [ ]:
import os, sys, subprocess, tempfile, importlib, json, yaml
from IPython.display import display, Markdown

if not os.path.isdir("st2"):
    subprocess.run(["git", "clone", "--depth", "1",
                     "https://github.com/caixiaoyao2025/st2.git"], check=True)
if "st2" not in sys.path:
    sys.path.insert(0, "st2")

from agent_connector.scanner import build_schema
from agent_connector.generator import generate_wiring, load_wrappers, load_adapter
from agent_connector.agent_preflight import preflight as run_preflight, display as pf_display
display(Markdown("**OK - all imports succeeded**"))

## Step 1 - Choose an agent

In [ ]:
import ipywidgets as widgets
from IPython.display import display

PRETESTED = {
    "Biomni (Stanford)": "https://github.com/snap-stanford/biomni.git",
    "BioChatter":        "https://github.com/biocypher/biochatter.git",
    "CellAgent":         "https://github.com/MlessS/CellAgent.git",
    "GeneAgent":         "https://github.com/MlessS/GeneAgent.git",
    "CRISPR-GPT":        "https://github.com/cong-lab/crispr-gpt-pub.git",
}

dropdown = widgets.Dropdown(
    options=["<custom URL>"] + list(PRETESTED.keys()),
    value="Biomni (Stanford)",
    description="Agent:",
    style={"description_width": "100px"},
)
custom_url = widgets.Text(
    value="", placeholder="https://github.com/user/repo.git",
    description="Custom URL:", style={"description_width": "100px"},
    layout=widgets.Layout(display="none"),
)

def on_change(change):
    custom_url.layout.display = "" if change["new"] == "<custom URL>" else "none"
dropdown.observe(on_change, names="value")

display(dropdown, custom_url)

In [ ]:
from IPython.display import display, Markdown

agent_url = PRETESTED.get(dropdown.value, custom_url.value)
if not agent_url:
    raise ValueError("No agent URL selected")

safe_name = dropdown.value.replace(" ", "_").lower().replace("(", "").replace(")", "")
clone_dir = f"/tmp/agent_{safe_name}"

if not os.path.isdir(os.path.join(clone_dir, ".git")):
    display(Markdown(f"Cloning `{agent_url}` ..."))
    r = subprocess.run(["git", "clone", "--depth", "1", agent_url, clone_dir],
                       capture_output=True, text=True, timeout=120)
    if r.returncode != 0:
        display(Markdown(f"**clone FAILED:** {r.stderr[:300]}"))
    else:
        display(Markdown(f"Cloned to `{clone_dir}`"))
else:
    display(Markdown(f"Already cloned: `{clone_dir}`"))

## Step 2 - Preflight: check agent environment

Scans the agent repo for dependencies (requirements.txt, pyproject.toml, imports),
checks what's installed, and auto-installs what's missing.

In [ ]:
from IPython.display import display, Markdown

pf = run_preflight(clone_dir, install_missing=False)
display(Markdown(pf_display(pf)))

if pf.status == "SETUP_REQUIRED" and pf.pip_installable:
    display(Markdown(f"**{len(pf.pip_installable)} packages can be auto-installed. Run next cell.**"))
elif pf.status == "READY":
    display(Markdown("**All dependencies satisfied!**"))

In [ ]:
from IPython.display import display, Markdown

if pf.status == "SETUP_REQUIRED" and pf.pip_installable:
    display(Markdown(f"Installing {len(pf.pip_installable)} packages ..."))
    pf = run_preflight(clone_dir, install_missing=True)
    display(Markdown(pf_display(pf)))
else:
    display(Markdown("Skipped (no auto-installable deps or already satisfied)"))

## Step 3 - Scan agent source & generate wiring

In [ ]:
import traceback
from IPython.display import display, Markdown

display(Markdown("Scanning agent source ..."))
try:
    schema = build_schema(clone_dir, include_evidence=False)
except Exception as e:
    display(Markdown(f"**build_schema FAILED:** {e}"))
    traceback.print_exc()
    raise

lines = [
    f"- **agent_class** = `{schema.get('agent_class')}`",
    f"- **reg_method** = `{schema.get('registration_method')}`",
    f"- **exec_method** = `{schema.get('execution_method')}`",
    f"- **wiring_style** = `{schema.get('wiring_style')}`",
]
display(Markdown("
".join(lines)))

reg_path = os.path.join("st2", "data", "mcp_registry.yaml")
tools = yaml.safe_load(open(reg_path, encoding="utf-8"))["tools"]
display(Markdown(f"**registry = {len(tools)} tools**"))

wiring_dir = tempfile.mkdtemp(prefix="wiring_")
try:
    wiring = generate_wiring(tools, schema, out_dir=wiring_dir)
except Exception as e:
    display(Markdown(f"**generate_wiring FAILED:** {e}"))
    traceback.print_exc()
    raise

display(Markdown(f"**wiring mode = {wiring['mode']}**"))
for k, v in wiring["artifacts"].items():
    if isinstance(v, str) and os.path.isfile(v):
        display(Markdown(f"- `artifact[{k}]` = `{v}`"))
    elif isinstance(v, list):
        display(Markdown(f"- `artifact[{k}]` = {len(v)} files"))
    else:
        display(Markdown(f"- `artifact[{k}]` = {str(v)[:200]}"))

if wiring["mode"] == "adapter":
    sys.path.insert(0, wiring_dir)

display(Markdown("**Instructions:**"))
display(Markdown(f"```
{wiring['instructions']}
```"))

## Step 4 - Inject tools

In [ ]:
from IPython.display import display, Markdown

inject_result = {"mode": wiring["mode"], "tools_injected": 0, "details": {}}

if wiring["mode"] == "adapter":
    if clone_dir not in sys.path:
        sys.path.insert(0, clone_dir)

    reg_style = schema.get("registration_style") or "function"
    try:
        wrappers = load_wrappers(
            package_name="generated_tools",
            registration_style=reg_style,
        )
    except Exception as e:
        display(Markdown(f"**load_wrappers FAILED:** {e}"))
        import traceback; traceback.print_exc()
        raise

    display(Markdown(f"Loaded **{len(wrappers)}** wrappers (style={reg_style})"))

    if schema.get("agent_class") in ("react", "ReflexionAgent") and "biomni" in agent_url.lower():
        try:
            from biomni.agent import A1
            old_stdout = sys.stdout
            sys.stdout = open(os.devnull, "w")
            try:
                agent = A1(path="/tmp/_biomni_data", expected_data_lake_files=[])
            finally:
                sys.stdout.close()
                sys.stdout = sys.__stdout__
            display(Markdown("A1 agent created"))
            for w in wrappers:
                try:
                    agent.add_tool(w)
                    inject_result["tools_injected"] += 1
                except Exception:
                    pass
            inject_result["details"]["agent"] = agent
            display(Markdown(f"Injected **{inject_result['tools_injected']}** tools via `add_tool`"))
        except ImportError as e:
            display(Markdown(f"Biomni import failed: `{e}` - using schema-level injection"))
            agent = type("FakeA1", (), {"module2api": {}})()
            agent.module2api = {"custom_tools": []}
            for w in wrappers:
                name = getattr(w, "name", None) or getattr(w, "__name__", "unknown")
                desc = getattr(w, "description", "")
                agent.module2api["custom_tools"].append({
                    "name": name, "description": desc, "module": "custom_tools",
                    "required_parameters": [], "parameters": {"type": "object", "properties": {}},
                })
                inject_result["tools_injected"] += 1
            inject_result["details"]["agent"] = agent
            display(Markdown(f"Injected **{inject_result['tools_injected']}** tools (schema-level)"))
    else:
        adapter_path = os.path.join(wiring_dir, "adapter.py")
        adapter_cls = load_adapter(schema.get("agent_class", "Agent"),
                                   adapter_path=adapter_path)
        agent = type("GenericAgent", (), {})()
        adapter = adapter_cls(agent)
        count = adapter.install_tools(wrappers)
        inject_result["tools_injected"] = count
        inject_result["details"]["agent"] = agent
        inject_result["details"]["adapter"] = adapter
        display(Markdown(f"Injected **{count}** tools via adapter"))

elif wiring["mode"] == "manifest":
    manifest = json.load(open(wiring["artifacts"]["manifest"], encoding="utf-8"))
    inject_result["tools_injected"] = len(manifest)
    inject_result["details"]["manifest"] = manifest
    display(Markdown(f"Manifest with **{len(manifest)}** tool schemas"))

elif wiring["mode"] == "config":
    config = yaml.safe_load(open(wiring["artifacts"]["config"], encoding="utf-8"))
    tool_count = len(config.get("tools", {}))
    inject_result["tools_injected"] = tool_count
    inject_result["details"]["config"] = config
    display(Markdown(f"Config with **{tool_count}** tools"))

elif wiring["mode"] == "prompt":
    prompt_block = wiring["artifacts"]["prompt_block"]
    inject_result["tools_injected"] = len(tools)
    inject_result["details"]["prompt_block"] = prompt_block
    display(Markdown(f"Prompt block: **{len(tools)}** tools, **{len(prompt_block)}** chars"))

display(Markdown(f"### Injection complete: **{inject_result['mode']}** mode, **{inject_result['tools_injected']}** tools"))

## Step 5 - Verify tool usage

In [ ]:
from IPython.display import display, Markdown

def verify_tool_call(tool_or_fn):
    try:
        if callable(tool_or_fn) and not hasattr(tool_or_fn, "run"):
            result = tool_or_fn(input="/dev/null")
        elif hasattr(tool_or_fn, "run"):
            result = tool_or_fn.run(input="/dev/null")
        elif hasattr(tool_or_fn, "call_with_args"):
            result = tool_or_fn.call_with_args({"input": "/dev/null"})
        else:
            return False, "tool is not callable"
        return True, str(result)[:300]
    except Exception as e:
        return True, f"[expected error] {type(e).__name__}: {e}"[:300]

rows = []
passed = failed = skipped = 0

if inject_result["mode"] == "adapter" and "agent" in inject_result["details"]:
    agent_obj = inject_result["details"]["agent"]
    for t in tools:
        name = t["name"]
        tool_obj = None
        if hasattr(agent_obj, "module2api"):
            for mod, entries in agent_obj.module2api.items():
                for entry in entries:
                    if entry.get("name") == name:
                        tool_obj = entry
                        break
        if hasattr(agent_obj, "tools") and isinstance(agent_obj.tools, dict):
            if name in agent_obj.tools:
                tool_obj = agent_obj.tools[name]
        if hasattr(agent_obj, "func2info") and name in agent_obj.func2info:
            tool_obj = agent_obj.func2info[name]
        if tool_obj is None:
            rows.append((name, "SKIP", "not found in agent"))
            skipped += 1
            continue
        has_schema = isinstance(tool_obj, dict) and ("name" in tool_obj or "description" in tool_obj)
        if not has_schema:
            rows.append((name, "PASS", "injected (schema-level)"))
            passed += 1
            continue
        ok, res = verify_tool_call(tool_obj)
        rows.append((name, "PASS" if ok else "FAIL", res[:80]))
        passed += ok
        failed += not ok

elif inject_result["mode"] == "manifest":
    for i, entry in enumerate(inject_result["details"]["manifest"]):
        fn = entry.get("function", {})
        name = fn.get("name", f"tool_{i}")
        ok = "parameters" in fn and bool(fn.get("description"))
        rows.append((name, "PASS" if ok else "FAIL", "valid OpenAI schema" if ok else "missing fields"))
        passed += ok
        failed += not ok

elif inject_result["mode"] == "config":
    for name, entry in inject_result["details"]["config"].get("tools", {}).items():
        ok = bool(entry.get("description"))
        rows.append((name, "PASS" if ok else "FAIL", "valid config" if ok else "missing description"))
        passed += ok
        failed += not ok

elif inject_result["mode"] == "prompt":
    block = inject_result["details"]["prompt_block"]
    for t in tools:
        name = t["name"]
        ok = f"### {name}" in block
        rows.append((name, "PASS" if ok else "FAIL", "in prompt block" if ok else "not found"))
        passed += ok
        failed += not ok

total = passed + failed + skipped
md_lines = [
    f"### Verification: {passed}/{total} passed, {failed} failed, {skipped} skipped",
    "",
    "| Status | Tool | Detail |",
    "|--------|------|--------|",
]
for name, status, detail in rows:
    icon = "OK" if status == "PASS" else ("!!" if status == "FAIL" else "--")
    md_lines.append(f"| {icon} | {name} | {detail[:60]} |")
md_lines.append("")
md_lines.append("**ALL CHECKS PASSED**" if failed == 0 else f"**WARNING: {failed} check(s) failed**")
display(Markdown("
".join(md_lines)))

## Step 6 - Dry-run: check tool commands in PATH

In [ ]:
import shutil, shlex, re
from IPython.display import display, Markdown

def render_cmd(template, args):
    filtered = {k: v for k, v in args.items() if v not in (None, "", False)}
    values = {k: str(v) for k, v in filtered.items()}
    tpl = re.sub(r"\{\{(\w+)\}\}", r"{\1}", template)
    tokens = shlex.split(tpl, posix=True)
    result = []
    for tok in tokens:
        m = re.match(r"^\{(\w+)\}$", tok)
        if m:
            result.append(values.get(m.group(1), ""))
        else:
            result.append(tok)
    return [t for t in result if t]

rows = []
for t in tools:
    name = t["name"]
    try:
        args = {}
        for pname, meta in (t.get("inputs") or {}).items():
            if meta.get("required"):
                pt = meta.get("type", "string")
                if pt in ("int", "integer"):
                    args[pname] = 1
                elif pt in ("float", "number"):
                    args[pname] = 0.1
                elif pt in ("bool", "boolean"):
                    args[pname] = False
                else:
                    args[pname] = "/dev/null"
        cmd = t.get("command", "")
        argv = render_cmd(cmd, args)
        cmd_name = argv[0] if argv else ""
        found = shutil.which(cmd_name)
        rows.append((name, "OK" if found else "NOT INSTALLED",
                      f"{cmd_name} -> {found}" if found else f"{cmd_name} not in PATH"))
    except Exception as e:
        rows.append((name, "FAIL", f"{type(e).__name__}: {e}"[:80]))

md_lines = [
    "### Dry-run: commands in PATH",
    "",
    "| Status | Tool | Detail |",
    "|--------|------|--------|",
]
for name, status, detail in rows:
    icon = "OK" if status == "OK" else ("!!" if status == "FAIL" else "--")
    md_lines.append(f"| {icon} | {name} | {detail[:60]} |")
display(Markdown("
".join(md_lines)))

## Summary

| Check | What it verifies |
|-------|-----------------|
| **Preflight** | Agent deps detected, installed, env vars checked |
| Schema validity | Wrapper has name, description, proper structure |
| Injection | Tool exists in agent's data structure after injection |
| Execution | Calling the tool doesn't crash the framework |
| Dry-run | Command exists in PATH (no auto-install)